# 12.11 先轉成文字會少掉什麼？

# 第 12 章：沿用相同介面加入聲音

前置：第10章模態展開與第7章loss mask。頻譜可獨立讀；聲音生成與log-mel都從PyTorch寫起，不依賴Whisper或torchaudio。FSDD爲額外真實語音pilot；本章默認合成音高／時長任務。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：形狀在圖片，音高在聲音：兩把鑰匙一起用**

替換每個模態，答案只改對應部分；保留新組合。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/joint.svg")))

**先想一想：**兩個音訊都轉成「你好」，還能從轉寫推斷哪個更高音嗎？

同一說詞可以用不同音高說出來。先轉文字會丟掉不在轉寫中的聲音屬性；直接features能保留，但模型仍需學會用它。

**把直覺寫成數學：**information(audio)一般比給定transcript包含更多屬性；不表示端到端一定更準。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
examples = [{"transcript": "你好", "pitch": "low"}, {"transcript": "你好", "pitch": "high"}]
print(examples)
assert examples[0]["transcript"] == examples[1]["transcript"]
print("純轉寫路徑缺音高資訊；直接feature路徑需額外音高監督")

**如何讀結果：**這是預錄轉寫／玩具標註示例，不調用ASR或外部模型。

**只改一件事：**只把音高標註也寫進文字，說明比較條件已變。
